In [10]:

# ============================================================
# NOTEBOOK 05 — NDCLD15 TRADITIONAL IMAGE AUGMENTATION
# Label-agnostic augmentation; no verified class labels assumed
# ============================================================
from pathlib import Path
from PIL import Image, ImageEnhance
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import json, random, time, zipfile

START = time.time()
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT = Path("/kaggle/working/nd_iris_project")
SOURCE_DIR = PROJECT / "notebook02_preprocessing" / "images_224"
OUT = PROJECT / "notebook05_traditional_augmentation"
AUG_DIR = OUT / "augmented_images"
AUG_DIR.mkdir(parents=True, exist_ok=True)

assert SOURCE_DIR.exists(), f"Missing preprocessed images: {SOURCE_DIR}"
sources = sorted(SOURCE_DIR.glob("*.png"))
assert sources, "No source PNG images found."

print(f"Source images: {len(sources):,}")
print("Creating 2 augmented variants per source image.")

records, errors = [], []

# Small, conservative transformations; no horizontal flip because
# eye orientation may carry information.
for src in tqdm(sources, desc="Traditional augmentation"):
    try:
        with Image.open(src) as im:
            original = im.convert("L").resize((224, 224), Image.Resampling.LANCZOS)

        variants = [
            # Variant 1: small rotation + mild brightness adjustment
            (
                original.rotate(
                    random.uniform(-5.0, 5.0),
                    resample=Image.Resampling.BILINEAR,
                    fillcolor=0
                ),
                random.uniform(0.95, 1.05),
                random.uniform(0.95, 1.05)
            ),
            # Variant 2: mild rotation + independent contrast/brightness
            (
                original.rotate(
                    random.uniform(-3.0, 3.0),
                    resample=Image.Resampling.BILINEAR,
                    fillcolor=0
                ),
                random.uniform(0.90, 1.10),
                random.uniform(0.95, 1.05)
            )
        ]

        for j, (rotated, brightness, contrast) in enumerate(variants, start=1):
            aug = ImageEnhance.Brightness(rotated).enhance(brightness)
            aug = ImageEnhance.Contrast(aug).enhance(contrast)
            out_name = f"{src.stem}_aug{j}.png"
            dst = AUG_DIR / out_name
            aug.save(dst, format="PNG")

            records.append({
                "source_filename": src.name,
                "augmented_filename": out_name,
                "augmented_path": str(dst),
                "variant": j,
                "rotation_degrees": round(
                    float(variants[j-1][0].getbbox() is not None), 0
                ) if False else None,
                "brightness_factor": round(brightness, 4),
                "contrast_factor": round(contrast, 4),
                "target_label": None,
                "label_status": "not available; no class assumed"
            })

    except Exception as e:
        errors.append({"source_filename": src.name, "error": str(e)[:300]})

manifest = pd.DataFrame(records)
manifest.to_csv(OUT / "NDCLD15_traditional_augmentation_manifest.csv", index=False)
pd.DataFrame(errors, columns=["source_filename", "error"]).to_csv(
    OUT / "NDCLD15_augmentation_errors.csv", index=False
)

# Verify output count, readability, dimensions, and mode.
verified, verification_errors = 0, []
for p in tqdm(sorted(AUG_DIR.glob("*.png")), desc="Verifying augmented images"):
    try:
        with Image.open(p) as im:
            assert im.size == (224, 224)
            assert im.mode == "L"
            im.verify()
        verified += 1
    except Exception as e:
        verification_errors.append({"filename": p.name, "error": str(e)[:300]})

pd.DataFrame(verification_errors, columns=["filename", "error"]).to_csv(
    OUT / "NDCLD15_augmentation_verification_errors.csv", index=False
)

source_count = len(sources)
expected = source_count * 2
actual = len(list(AUG_DIR.glob("*.png")))

summary = {
    "source_images": source_count,
    "variants_per_source": 2,
    "expected_augmented_images": expected,
    "augmented_images_created": actual,
    "manifest_rows": len(manifest),
    "verified_augmented_images": verified,
    "augmentation_errors": len(errors),
    "verification_errors": len(verification_errors),
    "output_size": [224, 224],
    "output_mode": "L",
    "transformations": [
        "small rotation within approximately +/-5 degrees",
        "brightness adjustment approximately 0.90 to 1.10",
        "contrast adjustment approximately 0.95 to 1.05"
    ],
    "labels_used": False,
    "note": "Augmentation outputs are not labeled by contact-lens class."
}

with open(OUT / "NDCLD15_augmentation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Small visual comparison grid.
preview_sources = sources[:8]
fig, axes = __import__("matplotlib.pyplot", fromlist=["pyplot"]).subplots(
    len(preview_sources), 3, figsize=(9, 3 * len(preview_sources))
)
plt = __import__("matplotlib.pyplot", fromlist=["pyplot"])
for i, src in enumerate(preview_sources):
    with Image.open(src) as im:
        axes[i, 0].imshow(im.convert("L"), cmap="gray")
    axes[i, 0].set_title("Original")
    for j in (1, 2):
        aug_path = AUG_DIR / f"{src.stem}_aug{j}.png"
        with Image.open(aug_path) as im:
            axes[i, j].imshow(im.convert("L"), cmap="gray")
        axes[i, j].set_title(f"Augmented {j}")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "NDCLD15_augmentation_preview.png", dpi=130)
plt.close()

# Write the notebook's preliminary conclusion report.
conclusion = f"""# Notebook 05 — Traditional Augmentation: Results

- Source images: {source_count:,}
- Expected augmented images: {expected:,}
- Augmented images created: {actual:,}
- Verified augmented images: {verified:,}
- Augmentation errors: {len(errors):,}
- Verification errors: {len(verification_errors):,}

Two conservatively transformed variants were generated per source image using
small rotations and mild brightness/contrast adjustments. No horizontal flips
were applied. No contact-lens labels were assumed or created.

The augmented images require visual review before being used in downstream
experiments. These transformations do not establish improved recognition
performance or preservation of class-specific features.
"""
(OUT / "Notebook05_Conclusion.md").write_text(conclusion)

# ZIP all results, including every augmented PNG.
ZIP_PATH = PROJECT / "ND_Iris_Notebook05_Results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, arcname=f"notebook05_traditional_augmentation/{p.relative_to(OUT)}")

with zipfile.ZipFile(ZIP_PATH) as z:
    bad = z.testzip()
    archived = z.namelist()
    archived_aug = sum(
        n.startswith("notebook05_traditional_augmentation/augmented_images/")
        and n.lower().endswith(".png")
        for n in archived
    )

print("\n" + "=" * 65)
print("NOTEBOOK 05 — TRADITIONAL AUGMENTATION RESULTS")
print("=" * 65)
print(json.dumps(summary, indent=2))
print(f"Archived files: {len(archived):,}")
print(f"Augmented PNGs in ZIP: {archived_aug:,}")
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"ZIP integrity: {'PASSED' if bad is None else 'FAILED: ' + str(bad)}")
print(f"Conclusion report: {OUT / 'Notebook05_Conclusion.md'}")
print(f"Results ZIP: {ZIP_PATH}")
print(f"Elapsed: {(time.time()-START)/60:.2f} minutes")

assert bad is None, "ZIP integrity check failed"
assert actual == expected, "Augmented image count differs from expected"
assert verified == actual, "Some augmented images failed verification"


Source images: 6,233
Creating 2 augmented variants per source image.


Traditional augmentation:   0%|          | 0/6233 [00:00<?, ?it/s]

Verifying augmented images:   0%|          | 0/12466 [00:00<?, ?it/s]


NOTEBOOK 05 — TRADITIONAL AUGMENTATION RESULTS
{
  "source_images": 6233,
  "variants_per_source": 2,
  "expected_augmented_images": 12466,
  "augmented_images_created": 12466,
  "manifest_rows": 12466,
  "verified_augmented_images": 12466,
  "augmentation_errors": 0,
  "verification_errors": 0,
  "output_size": [
    224,
    224
  ],
  "output_mode": "L",
  "transformations": [
    "small rotation within approximately +/-5 degrees",
    "brightness adjustment approximately 0.90 to 1.10",
    "contrast adjustment approximately 0.95 to 1.05"
  ],
  "labels_used": false,
  "note": "Augmentation outputs are not labeled by contact-lens class."
}
Archived files: 12,472
Augmented PNGs in ZIP: 12,466
ZIP size: 294.56 MB
ZIP integrity: PASSED
Conclusion report: /kaggle/working/nd_iris_project/notebook05_traditional_augmentation/Notebook05_Conclusion.md
Results ZIP: /kaggle/working/nd_iris_project/ND_Iris_Notebook05_Results.zip
Elapsed: 1.76 minutes


## Conclusion — Notebook 05: Traditional Image Augmentation

### Experimental Setup

Traditional augmentation was applied to the 6,233 preprocessed NDCLD15 grayscale images. Two augmented variants were generated for each source image using small rotations and mild brightness and contrast adjustments.

### Results

- **Original images:** 6,233
- **Augmented variants per image:** 2
- **Expected augmented images:** 12,466
- **Augmented images created:** 12,466
- **Successfully verified images:** 12,466
- **Augmentation errors:** 0
- **Verification errors:** 0
- **Image dimensions:** 224 × 224 pixels
- **Image format:** Grayscale PNG
- **Results ZIP integrity:** Passed

### Applied Transformations

- Small rotations within approximately ±5 degrees.
- Brightness adjustment within approximately 0.90–1.10.
- Contrast adjustment within approximately 0.95–1.05.
- No horizontal flipping was applied.

### Generated Artifacts

The notebook produced an augmentation manifest, error and verification reports, a JSON summary, a visual comparison preview, and the `ND_Iris_Notebook05_Results.zip` archive containing all generated results and augmented images.

### Limitations

Verified contact-lens class labels were unavailable. Therefore, no class labels were assigned to the augmented images. These transformations do not establish improved classification accuracy or guarantee preservation of all relevant ocular features. The generated images should be inspected before use in downstream experiments.

### Overall Conclusion

Notebook 05 successfully generated and verified **12,466 augmented images from 6,233 source images**, with zero reported processing or verification failures. The augmented dataset is ready for subsequent experimental use, subject to appropriate label validation and evaluation design.